<a href="https://colab.research.google.com/github/Buddhiimz/DeepLearning_Project/blob/feature%2Fjithma/IT22095176_DL_Assigment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import numpy as np
import tensorflow as tf
from PIL import Image
import matplotlib.pyplot as plt
import os
import shutil
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D


In [6]:
# Mount Google Drive to access datasets
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
# Extract train zip folders
import zipfile
with zipfile.ZipFile('/content/drive/MyDrive/Colab Notebooks/Assignment/train.zip', 'r') as zip_ref:
    zip_ref.extractall('train')

In [9]:
# Extract test zip folders
import zipfile
with zipfile.ZipFile('/content/drive/MyDrive/Colab Notebooks/Assignment/test.zip', 'r') as zip_ref:
    zip_ref.extractall('test')

In [10]:
import os

train_dir = '/content/train'
test_dir  = '/content/test'

print("Train root exists?", os.path.exists(train_dir))
print("Test root exists?", os.path.exists(test_dir))
print("\nTrain immediate subfolders (classes):")
print(sorted(os.listdir(train_dir))[:50])  # list class folders (or files)
print("\nTest immediate subfolders (classes):")
print(sorted(os.listdir(test_dir))[:50])

# Count images per class (quick)
for d in sorted(os.listdir(train_dir)):
    dpath = os.path.join(train_dir, d)
    if os.path.isdir(dpath):
        n = sum([len(files) for r, _, files in os.walk(dpath)])
        print(f"{d:20s} : {n} images")


Train root exists? True
Test root exists? True

Train immediate subfolders (classes):
['train']

Test immediate subfolders (classes):
['test']
train                : 2821 images


In [11]:
import matplotlib.pyplot as plt
from PIL import Image
import random

def show_samples(root, n_per_class=2):
    classes = sorted([d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d))])
    plt.figure(figsize=(12, 3 * len(classes)))
    i = 1
    for c in classes:
        files = [os.path.join(root, c, f) for f in os.listdir(os.path.join(root, c)) if f.lower().endswith(('jpg','png','jpeg'))]
        samples = random.sample(files, min(n_per_class, len(files)))
        for s in samples:
            img = Image.open(s).convert('RGB')
            plt.subplot(len(classes), n_per_class, i); plt.imshow(img); plt.axis('off'); plt.title(c)
            i += 1
    plt.tight_layout()

show_samples(train_dir, n_per_class=2)


<Figure size 1200x300 with 0 Axes>

In [12]:
import tensorflow as tf

image_size = (224, 224)
batch_size = 32
seed = 123

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    labels='inferred',
    label_mode='int',        # yields integer labels -> use SparseCategoricalLoss
    batch_size=batch_size,
    image_size=image_size,
    shuffle=True,
    validation_split=0.2,
    subset='training',
    seed=seed
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    labels='inferred',
    label_mode='int',
    batch_size=batch_size,
    image_size=image_size,
    shuffle=True,
    validation_split=0.2,
    subset='validation',
    seed=seed
)

# test dataset (no shuffling so predictions align with file order)
test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    labels='inferred',
    label_mode='int',
    batch_size=batch_size,
    image_size=image_size,
    shuffle=False
)

class_names = train_ds.class_names
num_classes = len(class_names)
print("Classes:", class_names)
print("Num classes:", num_classes)


Found 2820 files belonging to 1 classes.
Using 2256 files for training.
Found 2820 files belonging to 1 classes.
Using 564 files for validation.
Found 739 files belonging to 1 classes.
Classes: ['train']
Num classes: 1


In [13]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds   = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds  = test_ds.cache().prefetch(buffer_size=AUTOTUNE)


In [14]:
import tensorflow as tf
from tensorflow.keras import layers

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal_and_vertical'),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.2),
    layers.RandomBrightness(0.2)
], name='data_augmentation')

In [18]:
import tensorflow as tf
from tensorflow.keras.preprocessing import image_dataset_from_directory

train_dir = '/content/train'
test_dir = '/content/test'

# Create training and validation datasets
train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=(224, 224),
    batch_size=32,
    label_mode='categorical'
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=(224, 224),
    batch_size=32,
    label_mode='categorical'
)


Found 2820 files belonging to 1 classes.
Found 739 files belonging to 1 classes.


In [19]:
num_classes = len(train_ds.class_names)
print("Number of classes:", num_classes)

Number of classes: 1


In [20]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)


In [22]:
num_classes = len(train_ds.class_names)
print("Number of classes:", num_classes)

from tensorflow.keras import layers, models

base_model = tf.keras.applications.EfficientNetB3(
    include_top=False,
    weights='imagenet',
    input_shape=(224, 224, 3)
)
base_model.trainable = False  # start frozen

inputs = tf.keras.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
x = tf.keras.applications.efficientnet.preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.4)(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)
model.summary()



AttributeError: '_PrefetchDataset' object has no attribute 'class_names'

In [20]:
base_model.trainable = True
# freeze first N layers if desired:
for layer in base_model.layers[:100]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),  # small LR
              loss=tf.keras.losses.SparseCategoricalCrossentropy(),
              metrics=['accuracy'])
# then continue training


In [21]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

checkpoint_path = "/content/drive/MyDrive/Colab Notebooks/Assignment/best_model.h5"
callbacks = [
    ModelCheckpoint(checkpoint_path, monitor='val_accuracy', save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7)
]


In [22]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=callbacks
)


Epoch 1/15
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - accuracy: 0.0276 - loss: 3.9589
Epoch 1: val_accuracy improved from -inf to 0.06028, saving model to /content/drive/MyDrive/Colab Notebooks/Assignment/best_model.h5


71/71 ━━━━━━━━━━━━━━━━━━━━ 516s 7s/step - accuracy: 0.0277 - loss: 3.9586 - val_accuracy: 0.0603 - val_loss: 3.8779 - learning_rate: 1.0000e-05
Epoch 2/15
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - accuracy: 0.0802 - loss: 3.8040
Epoch 2: val_accuracy improved from 0.06028 to 0.11702, saving model to /content/drive/MyDrive/Colab Notebooks/Assignment/best_model.h5


71/71 ━━━━━━━━━━━━━━━━━━━━ 428s 6s/step - accuracy: 0.0803 - loss: 3.8038 - val_accuracy: 0.1170 - val_loss: 3.6936 - learning_rate: 1.0000e-05
Epoch 3/15
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - accuracy: 0.1161 - loss: 3.6533
Epoch 3: val_accuracy improved from 0.11702 to 0.18440, saving model to /content/drive/MyDrive/Colab Notebooks/Assignment/best_model.h5


71/71 ━━━━━━━━━━━━━━━━━━━━ 417s 6s/step - accuracy: 0.1163 - loss: 3.6532 - val_accuracy: 0.1844 - val_loss: 3.5213 - learning_rate: 1.0000e-05
Epoch 4/15
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - accuracy: 0.1915 - loss: 3.5121
Epoch 4: val_accuracy improved from 0.18440 to 0.25000, saving model to /content/drive/MyDrive/Colab Notebooks/Assignment/best_model.h5


71/71 ━━━━━━━━━━━━━━━━━━━━ 442s 6s/step - accuracy: 0.1916 - loss: 3.5117 - val_accuracy: 0.2500 - val_loss: 3.3527 - learning_rate: 1.0000e-05
Epoch 5/15
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - accuracy: 0.2403 - loss: 3.3556
Epoch 5: val_accuracy improved from 0.25000 to 0.29433, saving model to /content/drive/MyDrive/Colab Notebooks/Assignment/best_model.h5


...

71/71 ━━━━━━━━━━━━━━━━━━━━ 463s 7s/step - accuracy: 0.5461 - loss: 2.3913 - val_accuracy: 0.5319 - val_loss: 2.3545 - learning_rate: 1.0000e-05
Epoch 12/15
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - accuracy: 0.5665 - loss: 2.2893
Epoch 12: val_accuracy improved from 0.53191 to 0.56560, saving model to /content/drive/MyDrive/Colab Notebooks/Assignment/best_model.h5


71/71 ━━━━━━━━━━━━━━━━━━━━ 465s 7s/step - accuracy: 0.5664 - loss: 2.2893 - val_accuracy: 0.5656 - val_loss: 2.2365 - learning_rate: 1.0000e-05
Epoch 13/15
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 6s/step - accuracy: 0.6109 - loss: 2.1432
Epoch 13: val_accuracy improved from 0.56560 to 0.57979, saving model to /content/drive/MyDrive/Colab Notebooks/Assignment/best_model.h5


71/71 ━━━━━━━━━━━━━━━━━━━━ 448s 6s/step - accuracy: 0.6108 - loss: 2.1433 - val_accuracy: 0.5798 - val_loss: 2.1187 - learning_rate: 1.0000e-05
Epoch 14/15
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - accuracy: 0.6324 - loss: 2.0189
Epoch 14: val_accuracy improved from 0.57979 to 0.61525, saving model to /content/drive/MyDrive/Colab Notebooks/Assignment/best_model.h5


71/71 ━━━━━━━━━━━━━━━━━━━━ 414s 6s/step - accuracy: 0.6322 - loss: 2.0190 - val_accuracy: 0.6152 - val_loss: 2.0071 - learning_rate: 1.0000e-05
Epoch 15/15
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - accuracy: 0.6674 - loss: 1.9090
Epoch 15: val_accuracy improved from 0.61525 to 0.64007, saving model to /content/drive/MyDrive/Colab Notebooks/Assignment/best_model.h5


71/71 ━━━━━━━━━━━━━━━━━━━━ 442s 6s/step - accuracy: 0.6672 - loss: 1.9092 - val_accuracy: 0.6401 - val_loss: 1.8985 - learning_rate: 1.0000e-05


In [23]:
test_loss, test_acc = model.evaluate(test_ds)
print(f"Test loss: {test_loss:.4f}, Test accuracy: {test_acc:.4f}")


24/24 ━━━━━━━━━━━━━━━━━━━━ 66s 3s/step - accuracy: 0.5013 - loss: 2.3189
Test loss: 2.1190, Test accuracy: 0.5521
